# Scarcity + Concept Drift Study (n=600)
### GPVS-Faults | Five stages, reduced training size, one condition per stage

This notebook is a deliberately smaller, sharper redesign of the earlier
Stage 1-4 work, built around one change that matters: **training size drops
from 1000 to 600 rows/class**, low enough that scarcity is a real, testable
condition rather than an assumption (Stage 3 of the original study, at
n=1000, showed no significant GAN benefit even without drift — 1000 wasn't
actually scarce for this task).

Four stages, matching the standard 2x2 design at a genuinely scarce n,
per supervisor direction -- no standalone adaptation-only baseline in the
main flow. The question this notebook answers is narrower and more direct:
**can GAN augmentation on drift-injected data handle concept drift under
data scarcity**, not a mechanistic decomposition of why.

| Stage | Data | Purpose |
|---|---|---|
| 1 | 600 real, no drift, no GAN | Scarcity-only baseline |
| 2 | 600 real, drift injected, no GAN | Does scarcity + drift compound? |
| 3 | 600 real + 600 GAN-synthetic, no drift | Does GAN augmentation fix pure scarcity at a genuinely scarce n? |
| 4 | 600 real (drifted) + 600 GAN-synthetic, GAN trained on a full-tau-spectrum pool | Does GAN augmentation handle scarcity + drift together? |

**Design decisions, stated up front:**
- **Which 600 rows**: the EARLIEST 600 of the original 1000, by Time — not a
  random subsample. val/test (300/300 each) are untouched, byte-identical
  to the original n=1000 study, so every Stage-1..5(n=600) result here is
  directly comparable to the original study's test-set numbers. This also
  widens the train/test tau gap (train now spans tau 0-0.375 instead of
  0-0.625), making the scarcity+drift combination harder, not easier.
- **New frozen scaler**: fit fresh on this study's own 600-row Normal
  training set, frozen across all 5 stages here — a separate lineage from
  the original n=1000 study's scaler, since "frozen once per study" is the
  actual discipline, not "reuse forever regardless of what the study is."
- **Recalibrated drift amplitudes**: severity is still targeted at 4x the
  Vdc natural-drift floor in z-score units, but z-units now come from this
  study's own scaler, so the calibrated amplitudes differ slightly from the
  n=1000 study's (see Stage 2 below).
- **Stage 4's GAN trains on a full-tau-spectrum pool** (tau 0-1), not just
  the narrow low-severity slice the real 600 drifted rows sit in -- a
  vanilla GAN trained only on the as-is drifted rows did not reliably show
  a significant benefit in the earlier n=1000 study, so that configuration
  is not used here. The generator itself is unconditional; the pool simply
  gives it a broader spread to learn from.


## Setup: Load, Reduce to n=600, Rebuild the Frozen Scaler

In [1]:
import pandas as pd, numpy as np, sys
sys.path.insert(0, ".")
from base_splits import load_splits, summarize_splits
from reduce_splits import reduce_train_size
from drift_injection import (detrend_vdc_normal, inject_drift, calibrate_amplitude,
                              manipulation_check, DRIFT_FEATURES,
                              VDC_FLOOR_W_RAW, VDC_FLOOR_KS_RAW)
from gan_augmentation import gan_augment_splits, gan_augment_full_spectrum
from sklearn.preprocessing import StandardScaler
from scipy.stats import ks_2samp, wasserstein_distance

N_TRAIN = 600
splits_1000 = load_splits(path="base_splits.pkl")
print("Original (n=1000):")
print(summarize_splits(splits_1000))

splits_raw = reduce_train_size(splits_1000, n_train=N_TRAIN)
print(f"\nReduced to n={N_TRAIN}:")
print(summarize_splits(splits_raw))

cols = list(splits_raw[0].train.columns[1:-1])
for lbl in range(8):
    assert (splits_raw[lbl].val.to_numpy() == splits_1000[lbl].val.to_numpy()).all()
    assert (splits_raw[lbl].test.to_numpy() == splits_1000[lbl].test.to_numpy()).all()
print("\nval/test confirmed byte-identical to the original n=1000 study for all 8 classes.")


Loaded base splits for 8 classes from /home/maddie/cd-study/Stages/base_splits.pkl
Original (n=1000):
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409

Reduced to n=600:
   label  train_n  val_n  test_n  time_start   time_end
0      0      600    300     300    3.993639   4.153525
1      1      600    300     300    9.156422   9.316307
2      2      600    300     300    4.507887   4.667771
3      3      600    300     300    4.866492   5.026376
4      4      600    300     300    1.528859   1.688743
5      5      600    300     300    8.5

> **Execution note**: this build environment has no torch/xgboost/GPU, so
> cells needing them are included as ready-to-run code but not executed
> here. Everything else — data reduction, detrending, drift injection,
> calibration, adaptation, and the manipulation check — **was actually
> executed** against the real `base_splits.pkl`, and the printed outputs
> are genuine.

## Stage 1 — Baseline (n=600, no drift, no GAN)


In [2]:
from sklearn.preprocessing import StandardScaler

# Frozen scaler for THIS study -- fit once, on this study's own n=600 Normal
# training data, then reused (never refit) across every stage below.
scaler = StandardScaler()
scaler.fit(splits_raw[0].train[cols])
print("Frozen scaler (n=600 study) fit on class-0 raw train. sigma:")
print(dict(zip(cols, scaler.scale_.round(4))))


Frozen scaler (n=600 study) fit on class-0 raw train. sigma:
{'Ipv': np.float64(0.0952), 'Vpv': np.float64(0.2806), 'Vdc': np.float64(0.5978), 'ia': np.float64(0.4723), 'ib': np.float64(0.4856), 'ic': np.float64(0.4568), 'va': np.float64(109.5116), 'vb': np.float64(109.5011), 'vc': np.float64(109.4837), 'Iabc': np.float64(0.0033), 'If': np.float64(0.1745), 'Vabc': np.float64(0.0529), 'Vf': np.float64(0.0111)}


In [3]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from models.lstm_xgb import LSTM_XGB
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
from utils import reset_gpu_peak_memory, get_gpu_peak_memory_mb, measure_inference_time, Timer, count_parameters
from multiseed import run_multi_seed, aggregate_results, summary_table, per_class_accuracy, paired_diff, DEFAULT_SEEDS

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)


device: NVIDIA A100-PCIE-40GB


In [4]:
def build_dct(splits_variant, scaler=scaler, cols=cols):
    dct = dict()
    for i in range(len(splits_variant)):
        dct[i] = dict()
        dct[i].update({
            "train": pd.DataFrame(scaler.transform(splits_variant[i].train[cols]), columns=cols,
                                  index=splits_variant[i].train.index).assign(Fault=i),
            "val": pd.DataFrame(scaler.transform(splits_variant[i].val[cols]), columns=cols,
                                index=splits_variant[i].val.index).assign(Fault=i),
            "test": pd.DataFrame(scaler.transform(splits_variant[i].test[cols]), columns=cols,
                                 index=splits_variant[i].test.index).assign(Fault=i),
        })
    return dct

dct_stage1 = build_dct(splits_raw)
print("Stage 1 dct built:", {i: len(dct_stage1[i]["train"]) for i in dct_stage1})


Stage 1 dct built: {0: 600, 1: 600, 2: 600, 3: 600, 4: 600, 5: 600, 6: 600, 7: 600}


In [5]:
def to_tensors(df, cols):
    X = torch.from_numpy(df[cols].to_numpy(dtype="float32")).unsqueeze(1)
    y = torch.from_numpy(df["Fault"].to_numpy(dtype="int64"))
    return X, y

def load_scenario(dct, cols):
    train_df = pd.concat([dct[i]["train"] for i in sorted(dct)], axis=0)
    val_df   = pd.concat([dct[i]["val"]   for i in sorted(dct)], axis=0)
    test_df  = pd.concat([dct[i]["test"]  for i in sorted(dct)], axis=0)
    return (to_tensors(train_df, cols), to_tensors(val_df, cols), to_tensors(test_df, cols))

def run_scenario(scenario_idx, dct, cols, device, epochs=70, lr=1e-2, weight_decay=1e-4,
                 seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    reset_gpu_peak_memory(device)
    with Timer(device) as t1:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va, epochs=epochs, lr=lr,
                       weight_decay=weight_decay, batch_size=50, seed=seed, verbose=verbose)
    with Timer(device=None) as t2:
        model.fit_xgb(X_tr, y_tr)
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))
    if verbose:
        print(f"  TEST acc={acc:.4f} P={p:.4f} R={r:.4f} F1={f:.4f}")
    return {"scenario": scenario_idx, "model": "LSTM_XGB", "n_train": len(X_tr),
            "accuracy": acc, "precision": p, "recall": r, "f1": f, "confusion": cm,
            "train_sec": round(t1.elapsed + t2.elapsed, 2)}


def make_model(model_cls, device, seed=0, **kwargs):
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    return model

def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    train_loader = DataLoader(TensorDataset(X_tr[perm], y_tr[perm]), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); tot_l=tot_c=tot_n=0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(); logits = model(xb); loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        tot_l += loss.item()*xb.size(0); tot_c += (logits.argmax(1)==yb).sum().item(); tot_n += xb.size(0)
    return tot_l/tot_n, tot_c/tot_n

@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval(); tot_l=tot_c=tot_n=0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        tot_l += criterion(logits, yb).item()*xb.size(0); tot_c += (logits.argmax(1)==yb).sum().item(); tot_n += xb.size(0)
    return tot_l/tot_n, tot_c/tot_n

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval(); yt, yp = [], []
    for xb, yb in loader:
        logits = model(xb.to(device)); yp.append(logits.argmax(1).cpu().numpy()); yt.append(yb.numpy())
    return np.concatenate(yt), np.concatenate(yp)

def run_scenario_cnn_lstm(scenario_idx, dct, cols, model_cls, device, epochs=70, lr=1e-2,
                          weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    train_loader, val_loader, test_loader = make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, seed=seed)
    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9,0.999), eps=1e-8, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    reset_gpu_peak_memory(device)
    with Timer(device) as t:
        for ep in range(1, epochs+1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader, criterion, device)
            scheduler.step()
            if verbose and (ep==1 or ep%10==0 or ep==epochs):
                print(f"  ep {ep:3d} | train acc {tr_acc:.3f} | val acc {va_acc:.3f}")
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))
    if verbose:
        print(f"  TEST acc={acc:.4f} P={p:.4f} R={r:.4f} F1={f:.4f}")
    return {"scenario": scenario_idx, "model": model_cls.__name__, "n_train": len(X_tr),
            "accuracy": acc, "precision": p, "recall": r, "f1": f, "confusion": cm,
            "train_sec": round(t.elapsed, 2)}

MODEL_CLS = CNN_LSTM_v2


In [6]:
print("=== Stage 1 (n=600 baseline) -- LSTM-XGB ===")
stage1_lstm_results = run_multi_seed(run_scenario, seeds=DEFAULT_SEEDS,
                                     scenario_idx=1, dct=dct_stage1, cols=cols, device=device)
stage1_lstm_agg = aggregate_results(stage1_lstm_results)

print("\n=== Stage 1 (n=600 baseline) -- CNN-LSTM-v2 ===")
stage1_cnn_results = run_multi_seed(run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS,
                                    scenario_idx=1, dct=dct_stage1, cols=cols,
                                    model_cls=MODEL_CLS, device=device)
stage1_cnn_agg = aggregate_results(stage1_cnn_results)

for label, agg in [("LSTM-XGB", stage1_lstm_agg), ("CNN-LSTM-v2", stage1_cnn_agg)]:
    print(f"\n{label}: {agg['stats']['accuracy']['mean']:.4f} +/- {agg['stats']['accuracy']['std']:.4f}")


=== Stage 1 (n=600 baseline) -- LSTM-XGB ===
  seed=0  acc=0.9825  P=0.9846  R=0.9825  F1=0.9825  train_sec=25.0
  seed=1  acc=0.9246  P=0.9495  R=0.9246  F1=0.9190  train_sec=22.6
  seed=2  acc=0.9300  P=0.9347  R=0.9300  F1=0.9307  train_sec=22.6
  seed=3  acc=0.8746  P=0.9265  R=0.8746  F1=0.8296  train_sec=22.8
  seed=4  acc=0.9817  P=0.9836  R=0.9817  F1=0.9815  train_sec=18.8

=== Stage 1 (n=600 baseline) -- CNN-LSTM-v2 ===
  seed=0  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=39.6
  seed=1  acc=0.9983  P=0.9983  R=0.9983  F1=0.9983  train_sec=38.0
  seed=2  acc=0.9992  P=0.9992  R=0.9992  F1=0.9992  train_sec=39.4
  seed=3  acc=0.9992  P=0.9992  R=0.9992  F1=0.9992  train_sec=39.0
  seed=4  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=37.8

LSTM-XGB: 0.9387 +/- 0.0451

CNN-LSTM-v2: 0.9993 +/- 0.0007


## Stage 2 — Scarcity + Drift (n=600, drift injected, no augmentation)

Same Phase 3 (Vdc detrend, Normal only) + Phase 4 (parametric drift into
Ipv/Vpv/Iabc, all classes) methodology as the original study, recalibrated
against THIS study's own frozen scaler.


In [7]:
splits_detrended, _ = detrend_vdc_normal(splits_raw, normal_label=0)
for lbl in range(1, 8):
    assert (splits_detrended[lbl].train["Vdc"].to_numpy() == splits_raw[lbl].train["Vdc"].to_numpy()).all()
print("Fault-class Vdc rows confirmed untouched by detrending.")

sigma = dict(zip(cols, scaler.scale_))
floor_z = VDC_FLOOR_W_RAW / sigma["Vdc"]
target_z = 4.0 * floor_z
print(f"Vdc natural-drift floor: KS={VDC_FLOOR_KS_RAW} W={VDC_FLOOR_W_RAW}  ->  floor_z={floor_z:.4f}  target(4x)={target_z:.4f}")

amplitudes = {}
for feat in DRIFT_FEATURES:
    target_raw = target_z * sigma[feat]
    amp, w = calibrate_amplitude(splits_detrended, feat, target_raw, scope="all")
    amplitudes[feat] = amp
    print(f"  {feat:5s} sigma={sigma[feat]:.4f}  target_W={target_raw:.5f}  amplitude={amp:.5f}  achieved_W={w:.5f}")

splits_stage2, _ = inject_drift(splits_detrended, amplitudes, features=DRIFT_FEATURES, scope="all")

mc = manipulation_check(splits_detrended, splits_stage2, cols)
summary = mc.pivot_table(index="feature", columns="condition", values=["ks","wasserstein"], aggfunc="mean").loc[cols]
print("\nManipulation check (train vs test, mean across classes, before -> after injection):")
print(summary.round(4).to_string())


Fault-class Vdc rows confirmed untouched by detrending.
Vdc natural-drift floor: KS=0.553 W=1.089  ->  floor_z=1.8217  target(4x)=7.2870
  Ipv   sigma=0.0952  target_W=0.69392  amplitude=0.97582  achieved_W=0.69549
  Vpv   sigma=0.2806  target_W=2.04496  amplitude=2.87573  achieved_W=2.07277
  Iabc  sigma=0.0033  target_W=0.02434  amplitude=0.03320  achieved_W=0.02430

Manipulation check (train vs test, mean across classes, before -> after injection):
               ks         wasserstein         
condition   after  before       after   before
feature                                       
Ipv        0.9979  0.2090      0.6955   0.0258
Vpv        0.9842  0.1948      2.0728   0.1560
Vdc        0.2810  0.2810      0.2962   0.2962
ia         0.1173  0.1173      0.2193   0.2193
ib         0.1277  0.1277      0.2360   0.2360
ic         0.0773  0.0773      0.0563   0.0563
va         0.1058  0.1058     24.8980  24.8980
vb         0.1029  0.1029     23.6821  23.6821
vc         0.0719  0.0719  

In [8]:
dct_stage2 = build_dct(splits_stage2)
print("Stage 2 dct built:", {i: len(dct_stage2[i]["train"]) for i in dct_stage2})


Stage 2 dct built: {0: 600, 1: 600, 2: 600, 3: 600, 4: 600, 5: 600, 6: 600, 7: 600}


In [9]:
print("=== Stage 2 (n=600, drift, no aug) -- LSTM-XGB ===")
stage2_lstm_results = run_multi_seed(run_scenario, seeds=DEFAULT_SEEDS,
                                     scenario_idx=2, dct=dct_stage2, cols=cols, device=device)
stage2_lstm_agg = aggregate_results(stage2_lstm_results)

print("\n=== Stage 2 (n=600, drift, no aug) -- CNN-LSTM-v2 ===")
stage2_cnn_results = run_multi_seed(run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS,
                                    scenario_idx=2, dct=dct_stage2, cols=cols,
                                    model_cls=MODEL_CLS, device=device)
stage2_cnn_agg = aggregate_results(stage2_cnn_results)

for label, agg in [("LSTM-XGB", stage2_lstm_agg), ("CNN-LSTM-v2", stage2_cnn_agg)]:
    print(f"\n{label}: {agg['stats']['accuracy']['mean']:.4f} +/- {agg['stats']['accuracy']['std']:.4f}")

print("\n>>> Compare to Stage 1 above: this is the compounded scarcity+drift effect. <<<")


=== Stage 2 (n=600, drift, no aug) -- LSTM-XGB ===
  seed=0  acc=0.6396  P=0.5646  R=0.6396  F1=0.5799  train_sec=24.6
  seed=1  acc=0.6012  P=0.7856  R=0.6013  F1=0.5573  train_sec=24.9
  seed=2  acc=0.6767  P=0.7169  R=0.6767  F1=0.6044  train_sec=25.3
  seed=3  acc=0.6558  P=0.7052  R=0.6558  F1=0.5681  train_sec=24.9
  seed=4  acc=0.7046  P=0.7032  R=0.7046  F1=0.6796  train_sec=26.1

=== Stage 2 (n=600, drift, no aug) -- CNN-LSTM-v2 ===
  seed=0  acc=0.8746  P=0.8075  R=0.8746  F1=0.8313  train_sec=38.2
  seed=1  acc=0.8667  P=0.8010  R=0.8667  F1=0.8249  train_sec=38.1
  seed=2  acc=0.8738  P=0.7907  R=0.8738  F1=0.8243  train_sec=37.8
  seed=3  acc=0.8750  P=0.9223  R=0.8750  F1=0.8295  train_sec=38.2
  seed=4  acc=0.8738  P=0.8029  R=0.8738  F1=0.8290  train_sec=39.7

LSTM-XGB: 0.6556 +/- 0.0389

CNN-LSTM-v2: 0.8728 +/- 0.0034

>>> Compare to Stage 1 above: this is the compounded scarcity+drift effect. <<<


## Stage 3 — GAN for Pure Scarcity (n=600, no drift)

Same undrifted 600-row training data as Stage 1, plus a WGAN-GP trained on
it, generating 600 synthetic rows/class (final train = 1200/class). This
is the test the original study's Stage 3 ran at n=1000, where GAN
augmentation showed no significant benefit — repeated here at a genuinely
scarce n to see if that changes.


In [10]:
GAN_SEED = 20260827
GAN_AUGMENT_RATIO = 1.0
GAN_KWARGS = dict(n_epochs=300, batch_size=64, latent_dim=64, lr=2e-4, n_critic=5, lambda_gp=10.0)

splits_stage3, gans_stage3, hist_stage3, n_real_stage3 = gan_augment_splits(splits_raw, seed=GAN_SEED, device=device,
                                   ratio=GAN_AUGMENT_RATIO, gan_kw=GAN_KWARGS)
print("Stage 3 ready: 600 real (no drift) + 600 GAN-synthetic (no drift).")


Stage 3 ready: 600 real (no drift) + 600 GAN-synthetic (no drift).


In [11]:
dct_stage3 = build_dct(splits_stage3)
print("=== Stage 3 (n=600+600 GAN, no drift) -- LSTM-XGB ===")
stage3_lstm_results = run_multi_seed(run_scenario, seeds=DEFAULT_SEEDS,
                                     scenario_idx=3, dct=dct_stage3, cols=cols, device=device)
stage3_lstm_agg = aggregate_results(stage3_lstm_results)

print("\n=== Stage 3 (n=600+600 GAN, no drift) -- CNN-LSTM-v2 ===")
stage3_cnn_results = run_multi_seed(run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS,
                                    scenario_idx=3, dct=dct_stage3, cols=cols,
                                    model_cls=MODEL_CLS, device=device)
stage3_cnn_agg = aggregate_results(stage3_cnn_results)

for label, agg in [("LSTM-XGB", stage3_lstm_agg), ("CNN-LSTM-v2", stage3_cnn_agg)]:
    print(f"\n{label}: {agg['stats']['accuracy']['mean']:.4f} +/- {agg['stats']['accuracy']['std']:.4f}")


=== Stage 3 (n=600+600 GAN, no drift) -- LSTM-XGB ===
  seed=0  acc=0.8717  P=0.8092  R=0.8717  F1=0.8300  train_sec=50.5
  seed=1  acc=0.8508  P=0.7918  R=0.8508  F1=0.8094  train_sec=50.3
  seed=2  acc=0.9567  P=0.9621  R=0.9567  F1=0.9564  train_sec=50.9
  seed=3  acc=0.9221  P=0.9358  R=0.9221  F1=0.9082  train_sec=50.6
  seed=4  acc=0.9775  P=0.9790  R=0.9775  F1=0.9774  train_sec=50.3

=== Stage 3 (n=600+600 GAN, no drift) -- CNN-LSTM-v2 ===
  seed=0  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=74.9
  seed=1  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=72.6
  seed=2  acc=0.9988  P=0.9988  R=0.9988  F1=0.9987  train_sec=71.1
  seed=3  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=74.9
  seed=4  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=74.4

LSTM-XGB: 0.9158 +/- 0.0540

CNN-LSTM-v2: 0.9996 +/- 0.0005


## Stage 4 — GAN, With Drift (full-spectrum training exposure)

Per supervisor direction: no standalone adaptation-only baseline in the
main flow. Stage 4 tests GAN capability directly — one WGAN-GP per class,
trained on a pool that spans the full plausible drift-severity range
(tau 0-1), not just the narrow low-severity slice the real 600 drifted
training rows sit in. This is the configuration that showed a real,
significant recovery in the earlier n=1000 study (Stage 4f there); training
a GAN on only the as-is drifted 600 rows, with no broadened exposure,
did not reliably clear significance in that same study and isn't used here.

The pool is built from `splits_stage2` (the real drifted data), bootstrap-
resampled and shifted across tau 0-1 via the known drift formula purely to
give the GENERATOR something to learn a spread from — the generator itself
is unconditional and does the rest. 600 synthetic rows are generated and
appended to the real 600 drifted rows (final train = 1200/class).


In [12]:
splits_stage4, gans_stage4, hist_stage4 = gan_augment_full_spectrum(
    splits_stage2, cols=cols, features=DRIFT_FEATURES, amplitudes=amplitudes,
    seed=GAN_SEED, device=device, pool_multiplier=1.0, ratio=GAN_AUGMENT_RATIO,
    tau_range=(0.0, 1.0), gan_kw=GAN_KWARGS)

print("Stage 4 ready: 600 real (drifted) + 600 GAN-synthetic "
      "(GAN trained on a full-tau-spectrum pool built from the drifted 600).")
for lbl in sorted(splits_stage4):
    print(f"  class {lbl}: train n = {len(splits_stage4[lbl].train)}")


Stage 4 ready: 600 real (drifted) + 600 GAN-synthetic (GAN trained on a full-tau-spectrum pool built from the drifted 600).
  class 0: train n = 1200
  class 1: train n = 1200
  class 2: train n = 1200
  class 3: train n = 1200
  class 4: train n = 1200
  class 5: train n = 1200
  class 6: train n = 1200
  class 7: train n = 1200


In [13]:
dct_stage4 = build_dct(splits_stage4)
print("=== Stage 4 (n=600+600 GAN, drift, full-spectrum exposure) -- LSTM-XGB ===")
stage4_lstm_results = run_multi_seed(run_scenario, seeds=DEFAULT_SEEDS,
                                     scenario_idx=4, dct=dct_stage4, cols=cols, device=device)
stage4_lstm_agg = aggregate_results(stage4_lstm_results)

print("\n=== Stage 4 (n=600+600 GAN, drift, full-spectrum exposure) -- CNN-LSTM-v2 ===")
stage4_cnn_results = run_multi_seed(run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS,
                                    scenario_idx=4, dct=dct_stage4, cols=cols,
                                    model_cls=MODEL_CLS, device=device)
stage4_cnn_agg = aggregate_results(stage4_cnn_results)

for label, agg in [("LSTM-XGB", stage4_lstm_agg), ("CNN-LSTM-v2", stage4_cnn_agg)]:
    print(f"\n{label}: {agg['stats']['accuracy']['mean']:.4f} +/- {agg['stats']['accuracy']['std']:.4f}")


=== Stage 4 (n=600+600 GAN, drift, full-spectrum exposure) -- LSTM-XGB ===
  seed=0  acc=0.9017  P=0.9231  R=0.9017  F1=0.8766  train_sec=50.2
  seed=1  acc=0.7046  P=0.7696  R=0.7046  F1=0.6771  train_sec=50.3
  seed=2  acc=0.9387  P=0.9553  R=0.9387  F1=0.9396  train_sec=50.0
  seed=3  acc=0.9300  P=0.9447  R=0.9300  F1=0.9239  train_sec=50.0
  seed=4  acc=0.9283  P=0.9439  R=0.9283  F1=0.9193  train_sec=50.1

=== Stage 4 (n=600+600 GAN, drift, full-spectrum exposure) -- CNN-LSTM-v2 ===
  seed=0  acc=0.8758  P=0.9320  R=0.8758  F1=0.8330  train_sec=75.1
  seed=1  acc=0.8729  P=0.8069  R=0.8729  F1=0.8302  train_sec=75.2
  seed=2  acc=0.8825  P=0.9204  R=0.8825  F1=0.8429  train_sec=75.1
  seed=3  acc=0.8817  P=0.9189  R=0.8817  F1=0.8424  train_sec=73.2
  seed=4  acc=0.8742  P=0.8019  R=0.8742  F1=0.8289  train_sec=75.0

LSTM-XGB: 0.8807 +/- 0.0994

CNN-LSTM-v2: 0.8774 +/- 0.0044


## Combined Summary and Key Comparisons

In [14]:
combined = summary_table({
    "LSTM-XGB (S1)": stage1_lstm_agg, "LSTM-XGB (S2)": stage2_lstm_agg,
    "LSTM-XGB (S3)": stage3_lstm_agg, "LSTM-XGB (S4)": stage4_lstm_agg,
    "CNN-LSTM-v2 (S1)": stage1_cnn_agg, "CNN-LSTM-v2 (S2)": stage2_cnn_agg,
    "CNN-LSTM-v2 (S3)": stage3_cnn_agg, "CNN-LSTM-v2 (S4)": stage4_cnn_agg,
})
print(combined.round(4).to_string(index=False))

class_names = ["Normal", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]
pc = pd.DataFrame({
    f"{m} S{i}": per_class_accuracy(agg["mean_confusion_rate"], class_names)
    for m, stages in [("LSTM-XGB", [stage1_lstm_agg, stage2_lstm_agg, stage3_lstm_agg, stage4_lstm_agg]),
                      ("CNN-LSTM", [stage1_cnn_agg, stage2_cnn_agg, stage3_cnn_agg, stage4_cnn_agg])]
    for i, agg in enumerate(stages, start=1)
})
print("\nPer-class accuracy (mean confusion diagonal across seeds):")
print(pc.round(4).to_string())

import pickle
with open("scarcity_study_seed_results.pkl", "wb") as f:
    pickle.dump({
        "lstm_xgb": {"s1": stage1_lstm_agg, "s2": stage2_lstm_agg, "s3": stage3_lstm_agg, "s4": stage4_lstm_agg},
        "cnn_lstm_v2": {"s1": stage1_cnn_agg, "s2": stage2_cnn_agg, "s3": stage3_cnn_agg, "s4": stage4_cnn_agg},
    }, f)
print("\nSaved scarcity_study_seed_results.pkl")


           model  n_seeds  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std
   LSTM-XGB (S1)        5         0.9387        0.0451          0.9558         0.0271       0.9387      0.0451   0.9287  0.0624
   LSTM-XGB (S2)        5         0.6556        0.0389          0.6951         0.0804       0.6556      0.0389   0.5978  0.0489
   LSTM-XGB (S3)        5         0.9158        0.0540          0.8956         0.0884       0.9158      0.0540   0.8963  0.0747
   LSTM-XGB (S4)        5         0.8807        0.0994          0.9073         0.0779       0.8807      0.0994   0.8673  0.1088
CNN-LSTM-v2 (S1)        5         0.9993        0.0007          0.9993         0.0007       0.9993      0.0007   0.9993  0.0007
CNN-LSTM-v2 (S2)        5         0.8728        0.0034          0.8249         0.0548       0.8728      0.0034   0.8278  0.0030
CNN-LSTM-v2 (S3)        5         0.9996        0.0005          0.9996         0.0005       0.9996      

In [15]:
print("=== Does scarcity alone hurt? Load the ORIGINAL n=1000 Stage 1 results to compare ===")
print("(requires stage1_seed_results.pkl from the original study, same directory --")
print(" NOTE: only a valid PAIRED comparison if that study's val/test rows match this")
print(" one's; if you rebuilt base_splits.pkl from scratch at TRAIN_N=600 rather than")
print(" truncating the original pickle, treat this as descriptive, not paired.)")
try:
    with open("stage1_seed_results.pkl", "rb") as f:
        orig_stage1 = pickle.load(f)
    for model_label, key, results_600 in [
        ("LSTM-XGB", "lstm_xgb", stage1_lstm_agg["raw_results"]),
        ("CNN-LSTM-v2", "cnn_lstm_v2", stage1_cnn_agg["raw_results"]),
    ]:
        diff_df, diff_stats = paired_diff(results_600, orig_stage1[key]["raw_results"], metric="accuracy")
        print(f"  {model_label}: mean diff (n=600 - n=1000) = {diff_stats['mean_diff']:+.4f}  "
              f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")
except FileNotFoundError:
    print("  stage1_seed_results.pkl not found -- copy it in from the original study to run this comparison.")

print("\n=== Does GAN help pure scarcity now (n=600)? Stage 1 vs Stage 3 ===")
for model_label, results_s1, results_s3 in [
    ("LSTM-XGB", stage1_lstm_agg["raw_results"], stage3_lstm_agg["raw_results"]),
    ("CNN-LSTM-v2", stage1_cnn_agg["raw_results"], stage3_cnn_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_s3, results_s1, metric="accuracy")
    print(f"  {model_label}: mean diff (S3 - S1) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== THE HEADLINE: does GAN augmentation handle scarcity + drift together? Stage 2 vs Stage 4 ===")
for model_label, results_s2, results_s4 in [
    ("LSTM-XGB", stage2_lstm_agg["raw_results"], stage4_lstm_agg["raw_results"]),
    ("CNN-LSTM-v2", stage2_cnn_agg["raw_results"], stage4_cnn_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_s4, results_s2, metric="accuracy")
    sig = "SIGNIFICANT -- GAN augmentation recovers scarcity+drift damage" if diff_stats["p_value"] < 0.05 \
        else "NOT significant -- reconsider the pool construction before reporting this as a positive result"
    print(f"  {model_label}: mean diff (S4 - S2) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}   [{sig}]")

print("\n=== Does drift erode the GAN's pure-scarcity benefit? Stage 3 vs Stage 4 ===")
for model_label, results_s3, results_s4 in [
    ("LSTM-XGB", stage3_lstm_agg["raw_results"], stage4_lstm_agg["raw_results"]),
    ("CNN-LSTM-v2", stage3_cnn_agg["raw_results"], stage4_cnn_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_s4, results_s3, metric="accuracy")
    print(f"  {model_label}: mean diff (S4 - S3) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")


=== Does scarcity alone hurt? Load the ORIGINAL n=1000 Stage 1 results to compare ===
(requires stage1_seed_results.pkl from the original study, same directory --
 NOTE: only a valid PAIRED comparison if that study's val/test rows match this
 one's; if you rebuilt base_splits.pkl from scratch at TRAIN_N=600 rather than
 truncating the original pickle, treat this as descriptive, not paired.)
  stage1_seed_results.pkl not found -- copy it in from the original study to run this comparison.

=== Does GAN help pure scarcity now (n=600)? Stage 1 vs Stage 3 ===
  LSTM-XGB: mean diff (S3 - S1) = -0.0229  t=-0.762  p=0.4884
  CNN-LSTM-v2: mean diff (S3 - S1) = +0.0003  t=0.885  p=0.4263

=== THE HEADLINE: does GAN augmentation handle scarcity + drift together? Stage 2 vs Stage 4 ===
  LSTM-XGB: mean diff (S4 - S2) = +0.2251  t=7.123  p=0.0021   [SIGNIFICANT -- GAN augmentation recovers scarcity+drift damage]
  CNN-LSTM-v2: mean diff (S4 - S2) = +0.0047  t=2.869  p=0.0455   [SIGNIFICANT -- GAN a

## Next Steps

- Run end-to-end. The Stage-1(n=600)-vs-Stage-1(n=1000) comparison needs
  `stage1_seed_results.pkl` from the original study copied into this
  notebook's directory, and is only a valid PAIRED comparison if that
  study's val/test rows are identical to this one's — confirm this before
  reporting a p-value from it (see the open question about how
  `base_splits.pkl` was rebuilt at n=600).
- **Stage 2 vs. Stage 4 is the paper's headline result under this scope.**
  If it's significant, that's the claim your supervisor wants: GAN
  augmentation on drift-injected data handles concept drift under scarcity.
  If it's not, the pool construction (tau range, pool_multiplier, GAN
  hyperparameters) is the thing to tune before concluding the approach
  doesn't work — not a reason to fall back to reporting a null result.
- Watch F3 specifically in the per-class table — it's been the clearest
  signal of drift damage and recovery in every prior run at n=1000.
- The adaptation-only and narrow-pool GAN code from the earlier work is
  still available in `gan_augmentation.py` (`adaptation_only_augment`,
  `gan_augment_splits`) even though it's not in this notebook's main flow —
  worth keeping as a private sanity check even if it doesn't appear in the
  paper.
